# Document reading

In [ ]:
from pypdf import PdfReader
import numpy as np
import os

### Chunking


In [ ]:
import os

chunks = []

chunk_size = 500
overlap= 200

pdf_folder = "data"

for pdf_file in os.listdir(pdf_folder):

    if not pdf_file.endswith(".pdf"):
        continue

    pdf_path = os.path.join(pdf_folder, pdf_file)

    print("Processing:", pdf_file)

    reader = PdfReader(pdf_path)

    for page_num, page in enumerate(reader.pages):

        text = page.extract_text()

        if not text:
            continue

        start = 0

        while start < len(text):

            end = min(start + chunk_size, len(text))

            chunks.append({
                "chunk_id": len(chunks),
                "text": text[start:end],
                "page": page_num + 1,
                "source": pdf_file
            })

            start += chunk_size - overlap

# Embeddings

In [ ]:
from sentence_transformers import SentenceTransformer

In [ ]:
model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)
model

In [ ]:
chunk_texts = [chunk["text"] for chunk in chunks]
embeddings = model.encode(chunk_texts)

In [ ]:
embeddings.shape

In [ ]:
for i in range(len(chunks)):
    chunks[i]["embedding"] = embeddings[i]

In [ ]:
chunks[0].keys()

# Semantic search

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
def search_query(query,top_k=3):
    query_embedding = model.encode([query])

    similarities = cosine_similarity(
        query_embedding,embeddings
    )

    idx = np.argsort(similarities[0])[-top_k:][::-1]

    results = []
    for i in idx:
        results.append({
            "chunk_id":i,
            "score":similarities[0][i],
            "text":chunks[i]["text"]
        })
        
    return results

In [ ]:
results = search_query("what is graph?")
for r in results:
    print(r["text"])

# Optimizing chunking

In [ ]:
import faiss

In [ ]:
type(embeddings)

In [ ]:
embeddings.dtype

In [ ]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

In [ ]:
query = "What is bipartite graph?"
query_embedding = model.encode([query]).astype(np.float32)

In [ ]:
D,I = index.search(query_embedding,k=8)
print("Distances:")
print(D)

In [ ]:
print("Indices:")
print(I)

In [ ]:
for idx in I[0]:
    print(chunks[idx]["text"][:300])
    print("-"*50)
    print("\n")

In [ ]:
print(chunks[0].keys())

In [ ]:
lengths = [len(c["text"]) for c in chunks]

In [ ]:
print("Min: ",min(lengths))
print("Max: ",max(lengths))
print("Average: ",sum(lengths)/len(lengths))

# Turning into RAG system

In [ ]:
import google.generativeai as genai

from dotenv import load_dotenv
import os

load_dotenv()

API_KEY = os.getenv("GOOGLE_API_KEY")

genai.configure(api_key=API_KEY)

for m in genai.list_models():
    print(m.name)


In [ ]:
gemini_model = genai.GenerativeModel("gemini-2.5-flash")

In [ ]:
response = gemini_model.generate_content(
    "What is machine learning?"
)

In [ ]:
print(response.text)

### Buliding Context

In [ ]:
retrieved_chunks = []
for idx in I[0]:
    retrieved_chunks.append(
        chunks[idx]["text"]
    )

context = "\n\n".join(retrieved_chunks)


### Prompt Creating


In [ ]:
prompt = f"""
You are an expert teaching assistant.

Use only the information present in the context

If the answer is not present in the context,
reply with:
"I coud not find this information in the document."

Context:
{context}

Question:
{query}

Answer:
"""

In [ ]:
response = gemini_model.generate_content(
    prompt
)

In [ ]:
def ask_rag(query):
    query_embedding = model.encode(
        [query]
    ).astype(np.float32)

    D,I = index.search(query_embedding,k = 5)

    context = "\n\n".join(
        chunks[idx]["text"] for idx in I[0]
    )

    pages = sorted(
        set(chunks[idx]["page"] for idx in I[0])
    )
    
    sources = sorted(
        set(chunks[idx]["source"] for idx in I[0])
    )
    prompt = f"""
    You are a helpful assistant.
    
    Answer ONLY from the provided context.
    
    If the answer is not present in the context, say:
    
    "I could not find this information in the document."
    
    Context:
    {context}
    
    Question:
    {query}
    
    Answer:
    """

    response = gemini_model.generate_content(
        prompt
    )
    
    return {
        "answer": response.text,
        "pages": pages,
        "sources": sources
    }


In [ ]:
query = "what is graph?"

result = ask_rag(query)

print(result["answer"])
print("Pages:", result["pages"])
print("Sources:", result["sources"])

### saving embeddings and Faiss index


In [ ]:
faiss.write_index(index,"faiss_index.bin")

In [ ]:
import pickle
with open("chunks.pkl","wb") as f:
    pickle.dump(chunks,f)

In [ ]:
#load
index = faiss.read_index("faiss_index.bin")